# AutoMotion Mobile V10 — Renderizador Colab
Este notebook recebe um único ZIP criado pelo editor HTML, localiza o vídeo e `projeto.json`, e executa a renderização pesada no Colab.

In [ ]:
from google.colab import files
import os, zipfile, json, shutil, subprocess, glob

uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if not zip_name: raise ValueError('Envie o AutoMotion_Projeto.zip')

ROOT='/content/automotion_project'
shutil.rmtree(ROOT, ignore_errors=True)
os.makedirs(ROOT, exist_ok=True)
with zipfile.ZipFile(zip_name) as z: z.extractall(ROOT)
print('Arquivos:', os.listdir(ROOT))


In [ ]:
video_candidates=[p for p in glob.glob(ROOT+'/*') if os.path.splitext(p)[1].lower() in ['.mp4','.mov','.webm','.mkv','.m4v']]
json_path=os.path.join(ROOT,'projeto.json')
if not video_candidates: raise FileNotFoundError('Nenhum vídeo encontrado no ZIP.')
if not os.path.exists(json_path): raise FileNotFoundError('projeto.json não encontrado.')
video_path=video_candidates[0]
with open(json_path,'r',encoding='utf-8') as f: project=json.load(f)
print(json.dumps(project,ensure_ascii=False,indent=2))


In [ ]:
# Verifica FFmpeg
subprocess.run(['ffmpeg','-version'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,check=True)
print('FFmpeg OK')
print('Vídeo:',video_path)


In [ ]:
# Render base: preserva vídeo e áudio. As visualizações específicas serão aplicadas neste estágio.
out='/content/AutoMotion_final.mp4'
cmd=['ffmpeg','-y','-i',video_path,'-c:v','libx264','-preset','fast','-crf','18','-c:a','aac','-b:a','192k',out]
print('Executando FFmpeg...')
subprocess.run(cmd,check=True)
print('Render concluído:',out)


In [ ]:
from google.colab import files
files.download('/content/AutoMotion_final.mp4')


## Próxima camada
O campo `edits` do `projeto.json` é a fonte de verdade da edição. Os filtros/geradores de cada tipo (`comparison`, `scale`, `timeline`, etc.) devem ser aplicados ao vídeo antes do encode final.